# Heart Disease Classifier - Hackathon Showcase

This notebook is the presentation wrapper around the production scripts:
- `train.py`
- `evaluate.py`
- `predict.py`
- `app.py`

It runs training/evaluation on Colab GPU and shows final artifacts.

## 1) Runtime Setup (Colab GPU)
In Colab, first set: `Runtime -> Change runtime type -> GPU`.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Set your project directory in Drive
%cd /content/drive/MyDrive/BYTE2BEAT
!pip install -r requirements.txt

## 2) Verify Project Files and Config

In [ ]:
!ls -la
!python -c "import yaml, json; print(json.dumps(yaml.safe_load(open('config.yaml')), indent=2))"

In [ ]:
!python - << 'PY'
import os
from collections import Counter

base = 'dataset_split'
for split in ['train', 'test']:
    split_dir = os.path.join(base, split)
    if not os.path.exists(split_dir):
        print(f'Missing: {split_dir}')
        continue
    counts = {}
    for cls in sorted(os.listdir(split_dir)):
        cls_dir = os.path.join(split_dir, cls)
        if os.path.isdir(cls_dir):
            counts[cls] = sum(1 for f in os.listdir(cls_dir) if os.path.isfile(os.path.join(cls_dir, f)))
    print(split, counts)
PY

## 3) Train Model (uses GPU automatically if available)

In [ ]:
!python train.py --config config.yaml

## 4) Evaluate on Test Set

In [ ]:
!python evaluate.py --config config.yaml --checkpoint models/best_model.pth

## 5) Showcase Metrics and Artifacts

In [ ]:
import json
import pandas as pd
from PIL import Image
from IPython.display import display

with open('outputs/metrics.json', 'r', encoding='utf-8') as f:
    metrics = json.load(f)

print('Accuracy:', round(metrics['accuracy'], 4))
print('Macro F1:', round(metrics['macro_f1'], 4))
display(pd.DataFrame(metrics['classification_report']).T)

print('\nConfusion Matrix:')
display(Image.open('outputs/confusion_matrix.png'))

print('\nSample Predictions:')
display(pd.read_csv('outputs/sample_preds.csv'))

## 6) Single-Image Inference Demo

In [ ]:
# Replace this path with an actual test image
example_img = 'dataset_split/test/ASD/example.jpg'
!python predict.py --checkpoint models/best_model.pth --image "$example_img" --topk 3

## 7) Optional API Demo (FastAPI)
Run locally or in a separate environment for deployment demonstration:

```bash
uvicorn app:app --reload
```